In [ ]:
import h5py, numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import numpy.lib.recfunctions as rfn

from numba import njit
import numpy as np

In [ ]:
    # load
fname = 'threshold_input.hdf5'
with h5py.File('threshold_output.hdf5', 'w') as fout:
    for ig in range(1,9):
        with h5py.File(fname, 'r') as fh5:
            allhits = fh5[f'/io_group{ig}/hits']  # structured array with fields io_group,y,z,Q,?
            mask = np.isfinite(allhits['y_pix']) & np.isfinite(allhits['z_pix'])
            hits = allhits[mask]
            print(hits['channel_id'].max(), hits['channel_id'].min())
            print(hits['chip_id'].max(), hits['chip_id'].min())
            tile_id = 1+(hits['io_group']-1)*8+(hits['io_channel']-1)//4
            n0 = hits['channel_id'].astype(int).max()
            n1 = n0 * hits['chip_id'].astype(int).max()
            n2 = n1 * tile_id.astype(int).max() 
            uid =  (hits['io_group'].astype(int) * n2
                + tile_id.astype(int)*n1 + 
                hits['chip_id'].astype(int)*n0 +
                hits['channel_id'].astype(int))
            # uq =np.unique(uid,)
            # print(len(uq), 4900*8, 6400*8)
        
            # Step 1: prepare grouping keys
            y_i = np.rint(hits['y_pix'] * 1000).astype(int)
            z_i = np.rint(hits['z_pix'] * 1000).astype(int)
            io_group = hits['io_group'].astype(int)
        
            keys_df = pd.DataFrame({
                'io_group': io_group,
                'y_i' : y_i,
                'z_i' : z_i,
                # 'uid': uid,
                'Q': hits['Q'],
            })
        
            # Step 2: group and take median
            grouped = keys_df.groupby(['io_group', 'y_i', 'z_i'])['Q'].median().reset_index().sort_values(['io_group', 'y_i', 'z_i'], ascending=True)
            # grouped = keys_df.groupby(['io_group', 'uid'])['Q'].median().reset_index()
            grouped['io_group'] = np.rint(grouped['io_group']).astype('i4')
            # grouped['uid'] = np.rint(grouped['uid']).astype('i4')
            grouped['y_i'] = np.rint(grouped['y_i']).astype('i8')
            grouped['z_i'] = np.rint(grouped['z_i']).astype('i8')
            grouped = grouped[grouped['io_group'] == ig]
            # Convert to structured NumPy array
            structured = np.array(
                [tuple(row) for row in grouped.to_numpy()],
                dtype=[(col, grouped[col].dtype) for col in grouped.columns]
            )
        
            fout.create_dataset(f'io_group{ig}/threshold', data=structured)
            # Step 3: select io_group and plot
            qvals = grouped[grouped['io_group'] == ig]['Q']
            print('# negative threshold', (np.sum(qvals<1.5)), qvals.shape)
            hist, bins = np.histogram(qvals, bins=160, range=(-20,40))
            plt.bar(bins[:-1], hist, width=np.diff(bins))
            plt.title(f'io_group={ig}')
            plt.xlabel('median Q')
            plt.ylabel('counts')
            plt.show()

In [ ]:
# plotting
thresholds = []

with h5py.File('threshold_output.hdf5', 'r') as f:
    for ig in range(1,9):
        thres = f[f'/io_group{ig}/threshold'][:]
        if ig!=5 and ig!=6:
            spacing = 443.4
            npixels = 4900*8
            ygrid, zgrid = np.meshgrid(np.arange(4*70), np.arange(2*70), indexing='ij')
            arr = np.empty((70*4,70*2), dtype=thres.dtype)
        else:
            spacing = 388
            npixels = 6400*8
            ygrid, zgrid = np.meshgrid(np.arange(4*80), np.arange(2*80), indexing='ij')
            arr = np.empty((80*4,80*2), dtype=thres.dtype)
        yinds = np.rint((thres[:]['y_i'] - thres[0]['y_i']) / spacing).astype(int)
        zinds = np.rint((thres[:]['z_i'] - thres[0]['z_i']) / spacing).astype(int)
        print(ig, yinds.min(), yinds.max())
        print(zinds.min(), zinds.max())
        assert len(np.unique(np.vstack([yinds, zinds]).T, axis=1)) == len(thres)
        
        arr['Q'] = 1E16
        arr['y_i'] = thres['y_i'].min() - 100000
        arr['z_i'] = thres['z_i'].min() - 100000
        arr['io_group'] = ig
        arr[yinds, zinds] = thres

        thresholds.append(arr)
    # Step 3: select io_group and plot
with h5py.File('threshold_summary.hdf5', 'w') as f:
    for ig in range(1,9):
        f.create_dataset(f'io_group{ig}/threshold', data=thresholds[ig-1])
for ig in range(0,8,2):
    # print(thresholds[ig])
    thres = np.concatenate([thresholds[ig]['Q'], thresholds[ig+1]['Q']])
    hist, bins = np.histogram(thres, bins=300, range=(0,39))
    plt.bar(bins[:-1], hist, width=np.diff(bins))
    plt.title(f'Module={ig//2}; {len(thres)} channels triggered')
    plt.xlabel('threshold median Q')
    plt.ylabel('counts')
    plt.grid(True)
    plt.show()

In [ ]:
4900*8*2

In [ ]:
61882-61854